In [ ]:
import kagglehub
import os
import yaml
import shutil
import torch
from pathlib import Path
from ultralytics import YOLO

path = kagglehub.dataset_download("merfarukgnaydn/counter-strike-2-body-and-head-classification")
dataset_path = Path("/content/cs2_dataset")

if dataset_path.exists():
    shutil.rmtree(dataset_path)
shutil.copytree(path, dataset_path)

for cache in dataset_path.rglob("*.cache"):
    cache.unlink()

os.environ.pop("CUDA_VISIBLE_DEVICES", None)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

data_yaml = {
    "path":  str(dataset_path),
    "train": str(dataset_path / "train" / "images"),
    "val":   str(dataset_path / "val"   / "images"),
    "nc":    5,
    "names": ["none", "ct_body", "ct_head", "t_body", "t_head"],
}

yaml_path = "/content/cs2_dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

model = YOLO("yolov8n.pt")

model.train(
    data      = yaml_path,
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    name      = "cs2_headbody",
    project   = "/content/runs/detect",
    patience  = 10,
    optimizer = "AdamW",
    lr0       = 0.001,
    device    = device,
    workers   = 2,
    augment   = True,
    cache     = False,
    exist_ok  = True,
)

metrics = model.val()
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

shutil.copytree(
    "/content/runs/detect/cs2_headbody",
    "/content/drive/MyDrive/cs2_yolo_model",
    dirs_exist_ok=True
)
print("saved to MyDrive/cs2_yolo_model")